# Tree-based feature selection — random forest + permutation importance

**Goal:** identify relevant features fast and get to exploratory graphs with minimal
upfront cleaning.

**Data:** `data/dataset_train.csv` — ~11k rows from a Portuguese survey. The current export has 237 columns; the stored run used 238 (see the reproducibility note below).
The target is `AGE_RANGE` (16 ordered age bands). `data/dataset_test_X.csv` is the
unlabeled holdout for the later modeling step; it is **not touched** in this notebook.

**Strategy (playbook):**

| Step | Action |
|------|--------|
| 0 | Cheap pre-flight cuts: identifiers, survey weight, near-zero variance |
| 1 | Ordinal-encode every categorical; no scaling, no binning |
| 2 | One split; random forest (400 trees); permutation importance on the held-out set |
| 3 | Audit only the top-20 survivors: correlation, encoding, cardinality |
| 4 | Validate every cut with cross-validation on the training data |

References:
[Feature selection guide](https://scikit-learn.org/stable/modules/feature_selection.html) ·
[Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html) ·
[Correlated-features example](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance_multicollinear.html)

> **Reproducibility note.** The outputs stored in this notebook were produced from
> `data_v1/dataset_train.csv` (238 columns, including `WGT`). The current
> `data/dataset_train.csv` has 237 columns and no `WGT`. The pre-flight cell below now
> drops `WGT` only when it is present, so the notebook can run on either version; the
> stored model outputs have not been regenerated. Because `WGT` was never a model
> feature, the feature matrix should be unchanged, but rerun the notebook to confirm.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import category_encoders as ce

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import OrdinalEncoder
from IPython.display import display

RNG = 0                     # one seed everywhere, for reproducibility
TARGET = "AGE_RANGE"
ID_COLS = ["PID", "HHID"]    # unique per row -> pure identifiers, never features
WEIGHT_COL = "WGT"           # survey weight: inference metadata, not a predictor

df = pd.read_csv("../data/dataset_train.csv")
print(f"dataset_train.csv: {df.shape[0]:,} rows x {df.shape[1]} columns")

chance_level = df[TARGET].value_counts(normalize=True).max()
print(f"target: {TARGET} — {df[TARGET].nunique()} classes")
print(f"majority-class baseline accuracy: {chance_level:.3f}")
df.head(3).T.head(10)

In [ ]:
df.info()

In [ ]:
df.select_dtypes(include='int64')

In [ ]:
str_cols = df.select_dtypes(include="object").columns.tolist()  # your 233 columns

encoder = ce.BinaryEncoder(cols=str_cols)
df = encoder.fit_transform(df)

display(df.info())
df.iloc[:, 7:17].head()

## Step 0 — cheap pre-flight cuts

Fast passes that need no model:

1. **Identifiers** (`PID`, `HHID`): unique per row — they add noise and inflate importance.
2. **Survey weight** (`WGT`): sampling metadata, not a measurement on the person.
3. **Near-zero variance**: features whose second most common value covers < 0.1% of rows
   carry no usable signal (here: `PROXY`, `COUNTRY` are outright constant).

**Deliberately skipped here:** pruning one of each highly-correlated pair across all ~31k
feature pairs. For trees that is interpretability cleanup, not a requirement — the lighter
alternative from the playbook is to audit correlations **on the top-ranked features only**
(Step 3).

In [ ]:
y = df[TARGET].copy()
# `WGT` exists in data_v1 but was removed from the current data/ export.
# Drop it when present; keep the notebook runnable on both versions.
weight_cols = [WEIGHT_COL] if WEIGHT_COL in df.columns else []
if not weight_cols:
    print(f"Note: {WEIGHT_COL} is not in this data export; continuing without it.")
X = df.drop(columns=[TARGET] + ID_COLS + weight_cols)
dropped = list(ID_COLS) + weight_cols

def near_zero_variance(s, min_frac=0.001):
    """True if the feature is constant, or its 2nd most common value is < min_frac of rows."""
    vc = s.value_counts(dropna=False)
    return len(vc) == 1 or vc.iloc[1] / len(s) < min_frac

nzv = [c for c in X.columns if near_zero_variance(X[c])]
X = X.drop(columns=nzv)
dropped += nzv

print(f"Dropped at pre-flight ({len(dropped)}): {dropped}")
print(f"Features left: {X.shape[1]}  |  rows: {len(X):,}")

## Step 1 — encode categoricals

For a first pass, **ordinal-encode everything**. Do not spend time classifying each
categorical as nominal vs ordinal up front — accepted failure modes for pass one:

- arbitrary order near root splits (usually tolerable),
- fake proximity between levels (usually noise),
- inflated importance for high-cardinality features (**the main risk** — audited in Step 3).

High-cardinality rule: a nominal with more levels than ~10% of rows (> ~1,100 here) must be
dropped, grouped, or target-encoded with CV folds. The cell below checks whether any
survivor crosses that line.

Numerical features get **nothing** — no scaling, no binning; trees don't need it.
Missing values are a problem (scikit-learn rejects NaN): categoricals get an explicit
`__MISSING__` level (a tree can split on missingness itself), numerics get the median.

In [ ]:
n_rows = len(X)
card = X.nunique(dropna=False).sort_values(ascending=False)
high_card_threshold = 0.10 * n_rows
print(f"High-cardinality threshold (10% of {n_rows:,} rows): {high_card_threshold:.0f} levels")
print(card.head(10))
print(f"\nFeatures above threshold: {card[card > high_card_threshold].index.tolist() or 'none'}")

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print(f"{len(num_cols)} numeric, {len(cat_cols)} categorical")

X_enc = X.copy()
X_enc[cat_cols] = X[cat_cols].fillna("__MISSING__")
X_enc[num_cols] = X[num_cols].fillna(X[num_cols].median())

# Ordinal-encode every categorical for pass one; unseen levels at predict time map to -1.
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_enc[cat_cols] = encoder.fit_transform(X_enc[cat_cols])

print(f"Encoded matrix: {X_enc.shape}  |  remaining NaNs: {int(X_enc.isna().sum().sum())}")

## Step 2 — split, fit, rank

One hold-out split, `test_size=0.2`, stratified on the target. The forest is
`RandomForestClassifier(n_estimators=400)` — **not** a single tree (single-tree importance
is unstable and changes on every retrain).

Two sanity guards before trusting any ranking:

1. **Model sanity:** 5-fold `cross_val_score` on the training split. A near-random model
   produces meaningless importance — compare against the majority-class baseline.
2. **Ranking signal:** permutation importance computed **on the held-out set**, not the
   training set (training-set importance is inflated), and preferred over the built-in
   `feature_importances_`, which is more generous to high-cardinality accidents.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_enc, y, stratify=y, test_size=0.2, random_state=RNG
)
print(f"train: {X_tr.shape}  |  hold-out: {X_te.shape}")

rf = RandomForestClassifier(n_estimators=400, random_state=RNG, n_jobs=-1)
rf.fit(X_tr, y_tr)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=400, random_state=RNG, n_jobs=-1), X_tr, y_tr, cv=cv
)
print(f"5-fold CV accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
print(f"majority-class baseline: {chance_level:.3f}")
print(f"hold-out accuracy: {rf.score(X_te, y_te):.3f}")

In [ ]:
imp = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=RNG, n_jobs=-1)

perm = (
    pd.DataFrame(
        {
            "feature": X_enc.columns,
            "importance_mean": imp.importances_mean,
            "importance_std": imp.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False, ignore_index=True)
)
perm["rank"] = perm.index + 1
perm.head(25)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
top = perm.head(30).iloc[::-1]
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"], color="#4477aa")
ax.set_xlabel("Permutation importance (mean drop in hold-out accuracy)")
ax.set_title("Top 30 features by permutation importance")
fig.tight_layout()
plt.show()

## Step 3 — audit the survivors, not all 250 features

Three checks on the **top 20** only:

1. **Correlation.** If two ranked features correlate above ~0.9, their importance was split
   between them — the underlying signal is stronger than either row suggests. Conversely,
   if an expected-important feature ranks low, check whether a correlated duplicate
   absorbed its importance (shuffling one feature while its duplicate remains intact can
   hide a truly important feature).
2. **Encoding.** For each categorical in the top 20, is the ordinal code defensible?
   Re-encode the low-cardinality nominals with one-hot, refit, and check the importance
   holds — summing the dummy importances back onto the original feature, since one-hot
   dilutes importance across the dummies.
3. **Cardinality.** A top-ranked feature with hundreds of unique values is suspicious:
   verify the signal is real and not an overfit artifact (grouped splits / domain
   knowledge).

In [ ]:
TOP_K = 20
top_k = perm.head(TOP_K)["feature"].tolist()

corr = X_enc[top_k].corr()
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
pairs = corr.abs().where(mask).stack().sort_values(ascending=False)
print("Most-correlated pairs among the top 20:")
print(pairs.head(10).round(3))

high = pairs[pairs > 0.9]
print(f"\nPairs above 0.9 (importance split between them):")
print(high.round(3) if len(high) else "  none")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(top_k)), top_k, rotation=90)
ax.set_yticks(range(len(top_k)), top_k)
fig.colorbar(im, shrink=0.8)
ax.set_title("Correlation among top-20 features (encoded)")
fig.tight_layout()
plt.show()

In [ ]:
# One-hot re-encoding audit: which top-20 features are low-cardinality categoricals?
onehot_cols = [c for c in top_k if c in cat_cols and X[c].nunique(dropna=False) <= 15]
print(f"Re-encoding with one-hot: {onehot_cols}")

raw = X.copy()
raw[cat_cols] = raw[cat_cols].fillna("__MISSING__")
raw[num_cols] = raw[num_cols].fillna(raw[num_cols].median())

ordinal_rest = [c for c in cat_cols if c not in onehot_cols]
enc_oh = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_oh = raw.copy()
X_oh[ordinal_rest] = enc_oh.fit_transform(raw[ordinal_rest])
X_oh = pd.get_dummies(X_oh, columns=onehot_cols, prefix_sep="==", dtype=int)

# Same seed and row order => identical split, so the comparison is controlled.
X_oh_tr, X_oh_te, y_oh_tr, y_oh_te = train_test_split(
    X_oh, y, stratify=y, test_size=0.2, random_state=RNG
)
rf_oh = RandomForestClassifier(n_estimators=400, random_state=RNG, n_jobs=-1).fit(X_oh_tr, y_oh_tr)
imp_oh = permutation_importance(rf_oh, X_oh_te, y_oh_te, n_repeats=10, random_state=RNG, n_jobs=-1)

oh_tbl = pd.DataFrame({"feature": X_oh.columns, "importance_mean": imp_oh.importances_mean})
oh_tbl["original"] = oh_tbl["feature"].str.split("==", regex=False).str[0]
# One-hot dilutes importance across dummies — sum them back per original feature.
oh_sums = oh_tbl.groupby("original")["importance_mean"].sum()

compare = perm.head(TOP_K).set_index("feature")[["importance_mean"]].rename(
    columns={"importance_mean": "ordinal_encoded"}
)
compare["one_hot_sum"] = oh_sums.reindex(compare.index)
compare.round(4)

In [ ]:
audit = pd.DataFrame(
    {
        "feature": top_k,
        "kind": ["numeric" if c in num_cols else "categorical" for c in top_k],
        "n_levels": [int(X_enc[c].nunique()) for c in top_k],
        "perm_importance": perm.head(TOP_K)["importance_mean"].values,
    }
)
# Hundreds of unique values + high importance -> overfit artifact until proven otherwise.
audit["suspicious_cardinality"] = audit["n_levels"] > 100
audit.round(4)

## Step 4 — validate every cut

Compare **cross-validated performance of the reduced set against the full-feature
baseline**, fitted on the **training data only** — the hold-out set has already served its
single purpose (ranking) and is not reused to validate the cut.

Caveat (from the pitfall list): strictly leak-free selection would redo the ranking inside
each CV fold. Here the ranking signal came from one split's hold-out set — a pragmatic,
low-bias compromise at this scale, and the CV numbers quantify what the cut costs.

If an automatic cut is needed, wrap the forest in `SelectFromModel` with an explicit
threshold (below: keep the stronger half). Note it thresholds the built-in impurity
`feature_importances_`; permutation importance can be injected via `importance_getter`
when preferred.

In [ ]:
def cv_accuracy(cols):
    model = RandomForestClassifier(n_estimators=400, random_state=RNG, n_jobs=-1)
    scores = cross_val_score(
        model, X_tr[cols], y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=RNG)
    )
    return scores.mean()

results = pd.DataFrame(
    {
        "feature_set": ["full", "top-20"],
        "n_features": [X_tr.shape[1], len(top_k)],
        "cv_accuracy": [cv_accuracy(list(X_tr.columns)), cv_accuracy(top_k)],
    }
)
results

In [ ]:
sfm = SelectFromModel(
    RandomForestClassifier(n_estimators=400, random_state=RNG, n_jobs=-1),
    threshold="median",  # keep the stronger half of the features
).fit(X_tr, y_tr)
kept = X_tr.columns[sfm.get_support()].tolist()
print(f"SelectFromModel kept {len(kept)} of {X_tr.shape[1]} features")

results.loc[len(results)] = ["select-from-median", len(kept), cv_accuracy(kept)]
results

## Summary and next steps

**What the numbers should be read as:**

- `cv_accuracy` (full vs top-20 vs select-from-median): a small drop (or none) means the cut
  is safe; a large drop means the ranking discarded real signal — usually a correlated
  duplicate absorbed another feature's importance (Step 3.1).
- `one_hot_sum` vs `ordinal_encoded`: importances that hold up under re-encoding are real
  signal; importances that collapse were an ordinal-code artifact.
- `suspicious_cardinality`: verify flagged features with grouped splits or domain knowledge
  before trusting them.

**Pitfalls recap** (from the playbook):

- Selecting features on the test set is leakage: all selection happens inside training
  folds; the test set is touched once, at the end.
- Correlated features hide each other in permutation importance — scikit-learn documents
  this exact case with a hierarchical-clustering example ([link](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance_multicollinear.html)).
- Ordinal-encoding nominals is fine for trees but genuinely bad for linear models: if a
  later step uses regression or logistic regression, re-encode nominals with one-hot and
  drop one of each highly correlated pair.

**Next:** exploratory graphs (distributions vs `AGE_RANGE`, pairwise plots) for the
top-20 survivors only — no need to graph 230 features.

## Step 5 — minimal dataset for exploratory plots

This cell writes a **column subset** of the training data: the two IDs, the target,
and the 20 features that ranked highest in the permutation-importance run above.
It keeps **all 10,962 rows** so distributions and rare levels are not lost; the file
is smaller because it has 23 columns instead of 237.

The list is a snapshot of the stored run. It is for exploration, **not** a validated
feature set: Step 4 shows that the top-20 cut loses about 10% relative accuracy. See
`reports/01_feature_selection_review.md` for the full review.

The cell is self-contained and idempotent: it can run after a fresh kernel, and
running it again overwrites the same file with identical contents.

In [ ]:
from pathlib import Path
import pandas as pd

# --- Paths: work whether the notebook is launched from notebooks/ or the project root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "dataset_train.csv"
OUT_PATH = PROJECT_ROOT / "outputs" / "dataset_train_top20.csv"

TARGET = "AGE_RANGE"
ID_COLS = ["PID", "HHID"]
TOP20_FEATURES = [
    "HH_INACT", "HH_ACT", "HHNBPERS_0_13_COD", "MARSTALEGAL", "PA7_NAC",
    "MAINSTAT", "HA1D", "IN63", "HHNBPERS_COD", "HHINCOME", "LOCNACE_COD",
    "PA8_NAC", "PARTNERS", "HATLEVEL", "IN70", "PL1", "FT_PT", "IN54",
    "IN23", "SEX",
]
COLUMNS = ID_COLS + [TARGET] + TOP20_FEATURES

assert len(TOP20_FEATURES) == 20
assert len(COLUMNS) == len(set(COLUMNS)), "duplicate columns requested"

df = pd.read_csv(DATA_PATH, low_memory=False)
missing = [c for c in COLUMNS if c not in df.columns]
assert not missing, f"Missing columns in {DATA_PATH}: {missing}"

small = df.loc[:, COLUMNS].copy()
assert len(small) == len(df)
assert small.columns.is_unique

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
small.to_csv(OUT_PATH, index=False, encoding="utf-8")

print(f"source: {DATA_PATH} ({len(df):,} rows × {df.shape[1]} columns)")
print(f"wrote:  {OUT_PATH} ({len(small):,} rows × {small.shape[1]} columns, {OUT_PATH.stat().st_size / 1024:.0f} KiB)")
print(f"target classes: {small[TARGET].nunique()} | missing cells: {int(small.isna().sum().sum())}")
print("columns:", ", ".join(small.columns))

**How to use the file.** Load it with
`pd.read_csv("outputs/dataset_train_top20.csv")` from the project root (or
`../outputs/dataset_train_top20.csv` from `notebooks/`). Keep `"Não aplicável"` and
`"Sem resposta"` as explicit categories; they are survey answers, not missing values.
A good first plot is `AGE_RANGE` vs `HA1D` with the not-applicable level shown
separately: it makes the 55+ routing visible. The second is `AGE_RANGE` vs
`PA7_NAC` split by `SEX`, which shows why the female-only questions are really a
sex indicator plus a screening response.